In this notebook, numbers for the results section of the final paper are calculated. 

In [2]:
# Imports and display settings
import csv
import difflib
import re
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 80)

# Paths, agency set, shared helpers and data loading

In [17]:
# Paths, agency set, shared helpers and data loading
# Scope: PAR-based agencies only
DATA_DIR = Path("./../data/datasets")
MC_DIR = DATA_DIR / "manually_cleaned"
APPROVED_CHECK_DIR = MC_DIR / "approved_manually_cleaned"
EVAL_DIR = Path("./../evaluation/processed_files")

AGENCIES = ["EMA", "TGA", "PMDA", "Swissmedic"]  # order as in Table 2
MC_STEMS = {"EMA": "EMA", "TGA": "AUSTRALIA", "PMDA": "JAPAN", "Swissmedic": "SWISSMEDIC"}
EVAL_FILES = {
    "EMA": "20250717_EMA_gpt-4o_assessed.csv",
    "TGA": "20250717_TGA_gpt-4o_assessed.csv",
    "PMDA": "20250717_PMDA_gpt-4o_assessed.csv",
    "Swissmedic": "20250717_SwissMedic_gpt-4o_assessed.csv",
}

START_YEAR = 1995
APPROVAL_DECISIONS = {"approved", "conditional marketing authorisation", "conditional marketing authorization"}
PLACEHOLDERS = {"not reported", "not applicable", "na", "n/a", "tbd", "none", "nan", ""}
ERROR_PREFIX = "error: the response is not valid json"

RESULTS = {}  # collects all numbers for the summary cell at the end


def norm(s: pd.Series) -> pd.Series:
    """Lower-case, strip and collapse whitespace (missing values stay missing)."""
    return s.astype("string").str.replace(r"\s+", " ", regex=True).str.strip().str.lower()


def clean(s: pd.Series) -> pd.Series:
    """norm() plus placeholders and LLM extraction errors set to missing."""
    x = norm(s)
    return x.mask(x.isin(PLACEHOLDERS) | x.str.startswith(ERROR_PREFIX, na=False))


def split_multi(value) -> list:
    """Split a ';'-separated field into a de-duplicated list without placeholders."""
    if pd.isna(value):
        return []
    parts = [re.sub(r"\s+", " ", p).strip().lower() for p in str(value).split(";")]
    parts = [p for p in parts if p and p not in PLACEHOLDERS and not p.startswith(ERROR_PREFIX)]
    return list(dict.fromkeys(parts))


def pct(n, d, digits=1):
    """Percentage with commercial rounding (0.05 -> 0.1), as usually expected in manuscripts."""
    if not d:
        return np.nan
    q = Decimal(1).scaleb(-digits)
    return float((Decimal(int(n)) * 100 / Decimal(int(d))).quantize(q, rounding=ROUND_HALF_UP))


def n_pct(n, d, digits=1):
    """Format as '1,234 (12.3%)'."""
    return f"{int(n):,} ({pct(n, d, digits):.{digits}f}%)"


def to_date(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s.astype("string").str.strip(), format="%d.%m.%Y", errors="coerce")


def load_retrieved(agency: str) -> pd.DataFrame:
    df = pd.read_csv(MC_DIR / f"{MC_STEMS[agency]}_manually_cleaned.csv", low_memory=False)
    df.columns = df.columns.astype(str).str.strip()
    df["Agency"] = agency
    df["Decision_year_num"] = pd.to_numeric(df["Decision_year"], errors="coerce")
    df["Decision_clean"] = clean(df["Decision"])
    df["Extraction_error"] = (norm(df["Decision_year"]).str.startswith(ERROR_PREFIX, na=False)
                              | norm(df["Document_name"]).str.startswith("error:", na=False))
    return df


# retrieved PARs -> regulatory decisions >= 1995 -> approvals
retrieved = {a: load_retrieved(a) for a in AGENCIES}
df_retrieved = pd.concat(retrieved.values(), ignore_index=True)
df_decisions = df_retrieved[df_retrieved["Decision_year_num"] >= START_YEAR].copy()
df_approved = df_decisions[df_decisions["Decision_clean"].isin(APPROVAL_DECISIONS)].copy().reset_index(drop=True)

N_DEC = df_decisions["Agency"].value_counts().reindex(AGENCIES)
N_APPR = df_approved["Agency"].value_counts().reindex(AGENCIES)
N_APPR_ALL = {**N_APPR.to_dict(), "Overall": len(df_approved)}

# Consistency check against the exported approved datasets
for a in AGENCIES:
    exported = pd.read_csv(APPROVED_CHECK_DIR / f"{a}.csv", low_memory=False)
    same = set(exported["Document_name"]) == set(df_approved.loc[df_approved["Agency"] == a, "Document_name"])
    print(f"{a:<11} decisions >={START_YEAR}: {N_DEC[a]:>5,}   approvals: {N_APPR[a]:>5,}   "
          f"identical to approved_manually_cleaned/{a}.csv: {same}")
print(f"{'Total':<11} decisions >={START_YEAR}: {len(df_decisions):>5,}   approvals: {len(df_approved):>5,}")

EMA         decisions >=1995: 1,687   approvals: 1,491   identical to approved_manually_cleaned/EMA.csv: True
TGA         decisions >=1995: 1,042   approvals:   988   identical to approved_manually_cleaned/TGA.csv: True
PMDA        decisions >=1995:   408   approvals:   408   identical to approved_manually_cleaned/PMDA.csv: True
Swissmedic  decisions >=1995:   234   approvals:   233   identical to approved_manually_cleaned/Swissmedic.csv: True
Total       decisions >=1995: 3,371   approvals: 3,120


# Data completeness check

In [18]:
# Completeness check: are all downloaded PARs (document lists) contained in the analysed dataset?
DOC_LIST_DIR = Path("./../data/inference_data")  # {agency}.txt = full list of downloaded PARs
EVAL_LIST_DIR = Path("./../data/eval_data")      # eval_{agency}.txt = documents of the annotated sample


def read_list(path: Path) -> set:
    return {line.strip() for line in path.read_text(encoding="utf-8").splitlines() if line.strip()} if path.exists() else set()


rows = []
for a in AGENCIES:
    listed = read_list(DOC_LIST_DIR / f"{a}.txt")
    eval_docs = read_list(EVAL_LIST_DIR / f"eval_{a}.txt")
    names = retrieved[a]["Document_name"].astype(str)
    failed = names.str.lower().str.startswith("error:")  # LLM output not valid JSON -> document name lost
    in_data = set(names[~failed])
    missing = listed - in_data
    rows.append({
        "Agency": a,
        "PARs in document list": len(listed),
        "PARs in dataset": len(in_data),
        "Missing": len(missing),
        "  thereof evaluation sample": len(missing & eval_docs),
        "  thereof other": len(missing - eval_docs),
        "Rows with failed extraction (name lost)": int(failed.sum()),
        "In dataset but not in list": len(in_data - listed),
    })
    if missing - eval_docs:
        print(f"{a}: missing documents outside the evaluation sample: {sorted(missing - eval_docs)}")
completeness = pd.DataFrame(rows).set_index("Agency")
display(completeness)
if (completeness["Missing"] - completeness["Rows with failed extraction (name lost)"]).gt(0).any():
    print("WARNING: dataset is incomplete - fix before reporting numbers.")

EMA: missing documents outside the evaluation sample: ['alymsys-epar-public-assessment-report_en.pdf', 'avzivi-epar-public-assessment-report_en.pdf', 'ethyl-eicosapent-soft-gelatin-capsules-withdrawal-assessment-report_en.pdf', 'mvasi-epar-public-assessment-report_en.pdf', 'onbevzi-epar-public-assessment-report_en.pdf', 'vegzelma-epar-public-assessment-report_en.pdf']
TGA: missing documents outside the evaluation sample: ['auspar-riastap.pdf']


,PARs in document list,PARs in dataset,Missing,thereof evaluation sample,thereof other,Rows with failed extraction (name lost),In dataset but not in list
Agency,,,,,,,
EMA,1993,1987,7,1,6,6,1
TGA,1081,1050,31,30,1,0,0
PMDA,408,408,0,0,0,0,0
Swissmedic,234,234,0,0,0,0,0


# Results 3.2.1 / Supplementary Figure 1: study flow per agency

In [6]:
# Results 3.2.1 / Supplementary Figure 1: study flow per agency
def doc_type(name: str) -> str:
    name = str(name).lower()
    if "scientific-discussion" in name:
        return "EPAR scientific discussion"
    if "withdrawal-assessment" in name:
        return "withdrawal assessment report"
    if "public-assessment-report" in name:
        return "EPAR public assessment report"
    return "other"


rows = []
for a in AGENCIES:
    r = retrieved[a]
    year = r["Decision_year_num"]
    d = df_decisions[df_decisions["Agency"] == a]
    not_appr = d.loc[~d["Decision_clean"].isin(APPROVAL_DECISIONS), "Decision_clean"].fillna("not reported").value_counts()
    rows.append({
        "Agency": a,
        "Retrieved PARs": len(r),
        "Excluded: LLM output not valid JSON": int(r["Extraction_error"].sum()),
        "Excluded: decision year not reported": int((year.isna() & ~r["Extraction_error"]).sum()),
        f"Excluded: decision before {START_YEAR}": int((year < START_YEAR).sum()),
        f"Regulatory decisions (>= {START_YEAR})": len(d),
        "Excluded: not approved": int(not_appr.sum()),
        "   withdrawn": int(not_appr.get("withdrawn", 0)),
        "   refused / rejected": int(not_appr.get("refused", 0) + not_appr.get("rejected", 0)),
        "   decision not reported": int(not_appr.get("not reported", 0)),
        "Approvals (analysed)": int(N_APPR[a]),
    })
flow = pd.DataFrame(rows).set_index("Agency")
flow.loc["Total"] = flow.sum()
print("Study flow")
display(flow.T)
RESULTS["flow"] = flow

# Records lost before the >= 1995 filter: what are they?
lost = df_retrieved[df_retrieved["Decision_year_num"].isna()].copy()
lost["Doc_type"] = lost["Document_name"].map(doc_type)
lost["Decision_for_lost"] = lost["Decision_clean"].fillna("not reported / error")
print("\nRecords without usable decision year, by agency and extracted decision")
display(pd.crosstab(lost["Agency"], lost["Decision_for_lost"], margins=True))
print("\nEMA records without usable decision year, by document type")
display(lost.loc[lost["Agency"] == "EMA", "Doc_type"].value_counts())
RESULTS["lost_approved_missing_year"] = int((lost["Decision_clean"].isin(APPROVAL_DECISIONS)).sum())

Study flow


Agency,EMA,TGA,PMDA,Swissmedic,Total
Retrieved PARs,1993,1050,408,234,3685
Excluded: LLM output not valid JSON,6,0,0,0,6
Excluded: decision year not reported,300,8,0,0,308
Excluded: decision before 1995,0,0,0,0,0
Regulatory decisions (>= 1995),1687,1042,408,234,3371
Excluded: not approved,196,54,0,1,251
withdrawn,124,33,0,0,157
refused / rejected,72,21,0,1,94
decision not reported,0,0,0,0,0
Approvals (analysed),1491,988,408,233,3120



Records without usable decision year, by agency and extracted decision


Decision_for_lost,approved,conditional marketing authorisation,not reported / error,refused,withdrawn,All
Agency,,,,,,
EMA,246,2,13,8,37,306
TGA,0,0,0,0,8,8
All,246,2,13,8,45,314



EMA records without usable decision year, by document type


Doc_type
EPAR scientific discussion       268
withdrawal assessment report      29
other                              6
EPAR public assessment report      3
Name: count, dtype: int64

# Results 3.2.1 / Table 2 (top rows): regulatory decisions and approvals per agency

In [7]:
# Results 3.2.1 / Table 2 (top rows): regulatory decisions and approvals per agency
rows = []
for a in AGENCIES + ["Overall"]:
    d = df_decisions if a == "Overall" else df_decisions[df_decisions["Agency"] == a]
    p = df_approved if a == "Overall" else df_approved[df_approved["Agency"] == a]
    rows.append({
        "Agency": a,
        "Regulatory decisions": f"{len(d):,}",
        "Approvals [% of decisions]": n_pct(len(p), len(d)),
        "Share of all approvals": f"{pct(len(p), len(df_approved)):.1f}%",
        "First decision year": int(p["Decision_year_num"].min()),
        "Last decision year": int(p["Decision_year_num"].max()),
        "Latest decision date": to_date(p["Decision_date"]).max().date(),
    })
table2_counts = pd.DataFrame(rows).set_index("Agency")
display(table2_counts)
RESULTS["table2_counts"] = table2_counts

# Check: is the extracted decision year consistent with the extracted decision date?
dates = to_date(df_approved["Decision_date"])
mismatch = (dates.dt.year != df_approved["Decision_year_num"]) & dates.notna()
print(f"Approvals with decision year != year of decision date: {int(mismatch.sum())}; "
      f"approvals without valid decision date: {int(dates.isna().sum())}")

,Regulatory decisions,Approvals [% of decisions],Share of all approvals,First decision year,Last decision year,Latest decision date
Agency,,,,,,
EMA,"1,687","1,491 (88.4%)",47.8%,1995,2025,2025-02-27
TGA,"1,042",988 (94.8%),31.7%,2009,2025,2025-05-19
PMDA,408,408 (100.0%),13.1%,2007,2024,2024-08-01
Swissmedic,234,233 (99.6%),7.5%,2019,2024,2024-10-14
Overall,"3,371","3,120 (92.6%)",100.0%,1995,2025,2025-05-19


Approvals with decision year != year of decision date: 0; approvals without valid decision date: 52


# Results 3.2.2: annual number of approvals per agency

In [8]:
# Results 3.2.2: annual number of approvals per agency
year_tab = (
    df_approved.assign(Year=df_approved["Decision_year_num"].astype(int))
    .pivot_table(index="Year", columns="Agency", values="Document_name", aggfunc="count", fill_value=0)
    .reindex(columns=AGENCIES, fill_value=0)
)
year_tab["Total"] = year_tab.sum(axis=1)
display(year_tab)

last_date = to_date(df_approved["Decision_date"]).max()
last_complete_year = last_date.year - 1 if (last_date.month, last_date.day) != (12, 31) else last_date.year
all_four = year_tab[(year_tab[AGENCIES] > 0).all(axis=1)].index

RESULTS["annual_first"] = (int(year_tab.index.min()), int(year_tab["Total"].iloc[0]))
RESULTS["annual_peak"] = (int(year_tab["Total"].idxmax()), int(year_tab["Total"].max()))
RESULTS["annual_last_complete"] = (last_complete_year, int(year_tab.loc[last_complete_year, "Total"]))
RESULTS["annual_partial"] = (last_date.year, int(year_tab.loc[last_date.year, "Total"]), last_date.date())
RESULTS["years_all_four"] = (int(all_four.min()), int(all_four.max()))
RESULTS["first_year_per_agency"] = {a: int(year_tab.index[year_tab[a] > 0].min()) for a in AGENCIES}
RESULTS["last_year_per_agency"] = {a: int(year_tab.index[year_tab[a] > 0].max()) for a in AGENCIES}

print(f"First year {RESULTS['annual_first'][0]}: {RESULTS['annual_first'][1]} approvals")
print(f"Peak year {RESULTS['annual_peak'][0]}: {RESULTS['annual_peak'][1]} approvals")
print(f"Last complete year {last_complete_year}: {RESULTS['annual_last_complete'][1]} approvals")
print(f"{last_date.year} is partial (data until {last_date.date()}): {RESULTS['annual_partial'][1]} approvals")
print(f"First year per agency: {RESULTS['first_year_per_agency']}")
print(f"Last year per agency: {RESULTS['last_year_per_agency']}")
print(f"Years with data from all four agencies: {RESULTS['years_all_four'][0]}-{RESULTS['years_all_four'][1]}")

Agency,EMA,TGA,PMDA,Swissmedic,Total
Year,,,,,
1995,1,0,0,0,1
1996,10,0,0,0,10
1997,2,0,0,0,2
1998,4,0,0,0,4
1999,6,0,0,0,6
2000,3,0,0,0,3
2001,7,0,0,0,7
2002,3,0,0,0,3
2003,4,0,0,0,4


First year 1995: 1 approvals
Peak year 2022: 249 approvals
Last complete year 2024: 205 approvals
2025 is partial (data until 2025-05-19): 14 approvals
First year per agency: {'EMA': 1995, 'TGA': 2009, 'PMDA': 2007, 'Swissmedic': 2019}
Last year per agency: {'EMA': 2025, 'TGA': 2025, 'PMDA': 2024, 'Swissmedic': 2024}
Years with data from all four agencies: 2019-2024


# Results 3.2.2 / Table 2: disease classes 

In [9]:
# Results 3.2.2 / Table 2: disease classes (multi-label; % of approvals, so columns do not sum to 100%)
# DISEASE_CLASSES is defined in the cell "label-level scores for disease classes"
CLASS_FIXES = {
    "diseases of the blood and blood forming organs": "diseases of the blood and blood-forming organs",
    "endocrine, nutritional and metabolic diseases": "endocrine, nutritional, and metabolic diseases",
    "diseases of the musculskeletal system and connective tissue": "diseases of the musculoskeletal system and connective tissue",
    "congenital malformations and chromosal abnormalities": "congenital malformations and chromosomal abnormalities",
    "injury, poisining and certain other consequences of external causes": "injury, poisoning and certain other consequences of external causes",
}

df_approved["Disease_classes"] = df_approved["Disease_class(es)"].map(
    lambda v: list(dict.fromkeys(CLASS_FIXES.get(c, c) for c in split_multi(v))))

exploded = df_approved[["Agency", "Disease_classes"]].explode("Disease_classes").dropna()
dz = (exploded.groupby(["Disease_classes", "Agency"]).size().unstack(fill_value=0)
      .reindex(columns=AGENCIES, fill_value=0))
dz["Overall"] = dz.sum(axis=1)
dz = dz.sort_values("Overall", ascending=False)

table2_disease = dz.apply(lambda col: col.map(lambda v: n_pct(v, N_APPR_ALL[col.name])))
print("Disease classes: approvals with >= 1 mention, n (% of approvals)")
display(table2_disease)
RESULTS["disease_counts"] = dz

non_canonical = [c for c in dz.index if c not in DISEASE_CLASSES]
print(f"Classes not in the prompt list (check): {non_canonical}")

n_classes = df_approved["Disease_classes"].map(len)
print(f"\nNumber of disease classes per approval: {n_classes.value_counts().sort_index().to_dict()}")
print(f"Approvals with > 3 disease classes: {int((n_classes > 3).sum())}")
RESULTS["n_classes_per_approval"] = n_classes.value_counts().sort_index().to_dict()

# Overlap with neoplasms (effect of the prompt examples, e.g. lymphoma -> blood, lung cancer -> respiratory)
has = lambda c: df_approved["Disease_classes"].map(lambda lst: c in lst)
neo = has("neoplasms")
overlap = {}
for c in ["diseases of the blood and blood-forming organs", "diseases of the respiratory system",
          "diseases of the digestive system", "diseases of the skin", "diseases of the genitourinary system"]:
    k, both = int(has(c).sum()), int((has(c) & neo).sum())
    overlap[c] = (k, both)
    print(f"{c}: {k} approvals, thereof also neoplasms: {n_pct(both, k)}")
RESULTS["disease_overlap_neoplasms"] = overlap

Disease classes: approvals with >= 1 mention, n (% of approvals)


Agency,EMA,TGA,PMDA,Swissmedic,Overall
Disease_classes,,,,,
neoplasms,422 (28.3%),268 (27.1%),129 (31.6%),72 (30.9%),891 (28.6%)
infectious and parasitic diseases,226 (15.2%),177 (17.9%),79 (19.4%),39 (16.7%),521 (16.7%)
diseases of the blood and blood-forming organs,254 (17.0%),140 (14.2%),64 (15.7%),49 (21.0%),507 (16.3%)
"endocrine, nutritional, and metabolic diseases",212 (14.2%),118 (11.9%),47 (11.5%),33 (14.2%),410 (13.1%)
diseases of the respiratory system,163 (10.9%),126 (12.8%),55 (13.5%),27 (11.6%),371 (11.9%)
diseases of the nervous system,166 (11.1%),80 (8.1%),27 (6.6%),25 (10.7%),298 (9.6%)
diseases of the musculoskeletal system and connective tissue,115 (7.7%),97 (9.8%),38 (9.3%),9 (3.9%),259 (8.3%)
diseases of the circulatory system,135 (9.1%),81 (8.2%),23 (5.6%),14 (6.0%),253 (8.1%)
diseases of the digestive system,114 (7.6%),62 (6.3%),37 (9.1%),18 (7.7%),231 (7.4%)


Classes not in the prompt list (check): ['diseases of the immune system']

Number of disease classes per approval: {1: 1975, 2: 998, 3: 112, 4: 31, 5: 1, 6: 3}
Approvals with > 3 disease classes: 35
diseases of the blood and blood-forming organs: 507 approvals, thereof also neoplasms: 310 (61.1%)
diseases of the respiratory system: 371 approvals, thereof also neoplasms: 141 (38.0%)
diseases of the digestive system: 231 approvals, thereof also neoplasms: 107 (46.3%)
diseases of the skin: 227 approvals, thereof also neoplasms: 47 (20.7%)
diseases of the genitourinary system: 197 approvals, thereof also neoplasms: 89 (45.2%)


# Results 3.2.3 / Table 2: drug classes

In [10]:
# Results 3.2.3 / Table 2: drug classes
DRUG_CLASS_ORDER = ["small molecule", "biologics", "vaccine", "peptides and proteins", "other", "cell and gene therapy"]

df_approved["Drug_class_clean"] = clean(df_approved["Drug_class"]).fillna("not reported")
dcl = pd.crosstab(df_approved["Drug_class_clean"], df_approved["Agency"]).reindex(columns=AGENCIES, fill_value=0)
dcl["Overall"] = dcl.sum(axis=1)
dcl = dcl.reindex([c for c in DRUG_CLASS_ORDER if c in dcl.index] + [c for c in dcl.index if c not in DRUG_CLASS_ORDER])

table2_drug = dcl.apply(lambda col: col.map(lambda v: n_pct(v, N_APPR_ALL[col.name])))
print("Drug classes, n (% of approvals)")
display(table2_drug)
RESULTS["drug_counts"] = dcl

# Proportion of drug classes over time (periods; early years contain almost only EMA)
PERIOD_BINS = [1994, 2009, 2014, 2019, 2025]
PERIOD_LABELS = ["1995-2009", "2010-2014", "2015-2019", "2020-2025"]
df_approved["Period"] = pd.cut(df_approved["Decision_year_num"], bins=PERIOD_BINS, labels=PERIOD_LABELS)

by_period = pd.crosstab(df_approved["Period"], df_approved["Drug_class_clean"], normalize="index").mul(100).round(1)
by_period = by_period.reindex(columns=[c for c in DRUG_CLASS_ORDER if c in by_period.columns])
by_period["n"] = df_approved["Period"].value_counts().reindex(by_period.index)
print("\nDrug class distribution by period (% of approvals in period)")
display(by_period)
RESULTS["biologics_by_period"] = by_period["biologics"].to_dict()

Drug classes, n (% of approvals)


Agency,EMA,TGA,PMDA,Swissmedic,Overall
Drug_class_clean,,,,,
small molecule,919 (61.6%),507 (51.3%),226 (55.4%),112 (48.1%),"1,764 (56.5%)"
biologics,350 (23.5%),318 (32.2%),125 (30.6%),73 (31.3%),866 (27.8%)
vaccine,68 (4.6%),89 (9.0%),30 (7.4%),20 (8.6%),207 (6.6%)
peptides and proteins,89 (6.0%),57 (5.8%),22 (5.4%),9 (3.9%),177 (5.7%)
other,38 (2.5%),14 (1.4%),5 (1.2%),9 (3.9%),66 (2.1%)
cell and gene therapy,27 (1.8%),3 (0.3%),0 (0.0%),10 (4.3%),40 (1.3%)



Drug class distribution by period (% of approvals in period)


Drug_class_clean,small molecule,biologics,vaccine,peptides and proteins,other,cell and gene therapy,n
Period,,,,,,,
1995-2009,66.2,20.8,5.5,5.2,1.9,0.3,308
2010-2014,67.4,19.6,5.2,5.1,2.2,0.5,769
2015-2019,56.9,32.0,3.1,5.2,1.9,0.9,900
2020-2025,46.4,31.8,10.7,6.6,2.3,2.4,1143


# Results 3.2.4: administration routes

In [11]:
# Results 3.2.4: administration routes
# Denominator = approvals (a decision can have several routes). Parenteral as defined in the manuscript: IV, SC, IM.
PARENTERAL = {"intravenous", "subcutaneous", "deep subcutaneous", "intramuscular"}

df_approved["Routes"] = df_approved["Administration_route"].map(split_multi)
df_approved["Route_oral"] = df_approved["Routes"].map(lambda r: "oral" in r)
df_approved["Route_parenteral"] = df_approved["Routes"].map(lambda r: bool(PARENTERAL & set(r)))

rows = []
for a in AGENCIES + ["Overall"]:
    sub = df_approved if a == "Overall" else df_approved[df_approved["Agency"] == a]
    n = len(sub)
    rows.append({
        "Agency": a,
        "Oral": n_pct(sub["Route_oral"].sum(), n),
        "Parenteral (IV/SC/IM)": n_pct(sub["Route_parenteral"].sum(), n),
        "Oral and parenteral": n_pct((sub["Route_oral"] & sub["Route_parenteral"]).sum(), n),
        "Other routes only": n_pct((~sub["Route_oral"] & ~sub["Route_parenteral"] & sub["Routes"].map(len).gt(0)).sum(), n),
        "Route not reported": n_pct(sub["Routes"].map(len).eq(0).sum(), n),
    })
routes_tab = pd.DataFrame(rows).set_index("Agency")
print("Administration routes, n (% of approvals)")
display(routes_tab)

o = df_approved
RESULTS["route_oral"] = (int(o["Route_oral"].sum()), pct(o["Route_oral"].sum(), len(o)))
RESULTS["route_parenteral"] = (int(o["Route_parenteral"].sum()), pct(o["Route_parenteral"].sum(), len(o)))

# For comparison: old method (share of route mentions instead of approvals)
mentions = o["Routes"].explode().dropna()
print(f"\n(Old method, % of {len(mentions):,} route mentions) oral {pct((mentions == 'oral').sum(), len(mentions))}%, "
      f"parenteral {pct(mentions.isin(PARENTERAL).sum(), len(mentions))}%")

# Parenteral use by drug class
par_by_class = o.groupby("Drug_class_clean").agg(n=("Route_parenteral", "size"), parenteral=("Route_parenteral", "sum"),
                                                 oral=("Route_oral", "sum"))
par_by_class["% parenteral"] = (100 * par_by_class["parenteral"] / par_by_class["n"]).round(1)
par_by_class["% oral"] = (100 * par_by_class["oral"] / par_by_class["n"]).round(1)
print("\nRoutes by drug class")
display(par_by_class.reindex([c for c in DRUG_CLASS_ORDER if c in par_by_class.index]))
advanced = o["Drug_class_clean"].isin(["biologics", "peptides and proteins", "cell and gene therapy"])
RESULTS["parenteral_biologics_adv"] = (int(o.loc[advanced, "Route_parenteral"].sum()), int(advanced.sum()))

# Stability over time
route_period = o.groupby("Period", observed=True)[["Route_oral", "Route_parenteral"]].mean().mul(100).round(1)
print("\nRoutes by period (% of approvals)")
display(route_period)

Administration routes, n (% of approvals)


,Oral,Parenteral (IV/SC/IM),Oral and parenteral,Other routes only,Route not reported
Agency,,,,,
EMA,696 (46.7%),637 (42.7%),14 (0.9%),145 (9.7%),27 (1.8%)
TGA,390 (39.5%),494 (50.0%),9 (0.9%),113 (11.4%),0 (0.0%)
PMDA,188 (46.1%),190 (46.6%),0 (0.0%),29 (7.1%),1 (0.2%)
Swissmedic,95 (40.8%),124 (53.2%),0 (0.0%),14 (6.0%),0 (0.0%)
Overall,"1,369 (43.9%)","1,445 (46.3%)",23 (0.7%),301 (9.6%),28 (0.9%)



(Old method, % of 3,200 route mentions) oral 42.8%, parenteral 47.3%

Routes by drug class


,n,parenteral,oral,% parenteral,% oral
Drug_class_clean,,,,,
small molecule,1764,258,1333,14.6,75.6
biologics,866,795,3,91.8,0.3
vaccine,207,186,2,89.9,1.0
peptides and proteins,177,150,10,84.7,5.6
other,66,26,21,39.4,31.8
cell and gene therapy,40,30,0,75.0,0.0



Routes by period (% of approvals)


,Route_oral,Route_parenteral
Period,,
1995-2009,54.2,35.1
2010-2014,50.8,35.5
2015-2019,43.2,48.0
2020-2025,36.9,55.3


# Results 3.2.5: pharmaceutical forms by drug class

In [12]:
# Results 3.2.5: pharmaceutical forms by drug class
# Denominator = approvals within the drug class (a decision can have several forms, e.g. powder + solvent)
FORM_GROUPS = {
    "tablet / capsule": {"tablet", "capsule"},
    "solution / injection": {"solution", "injection", "concentrate", "infusion", "pre-filled syringe", "syringe",
                             "lyophilized injection"},
    "powder / lyophilisate": {"powder", "lyophilisate", "lyophilized preparation"},
    "suspension": {"suspension"},
    "emulsion / dispersion": {"emulsion", "dispersion"},
}
DRUG_CLASS_GROUPS = {
    "Small molecules": {"small molecule"},
    "Biologics": {"biologics"},
    "Peptides and proteins": {"peptides and proteins"},
    "Cell and gene therapy": {"cell and gene therapy"},
    "Biologics + peptides/proteins + CGT": {"biologics", "peptides and proteins", "cell and gene therapy"},
    "Vaccines": {"vaccine"},
}

df_approved["Forms"] = df_approved["Pharmaceutical_form"].map(split_multi)

rows, top_raw = [], {}
for label, classes in DRUG_CLASS_GROUPS.items():
    sub = df_approved[df_approved["Drug_class_clean"].isin(classes)]
    n = len(sub)
    row = {"Drug class": label, "n approvals": n}
    for fg, members in FORM_GROUPS.items():
        row[fg] = n_pct(sub["Forms"].map(lambda f: bool(members & set(f))).sum(), n)
    rows.append(row)
    top_raw[label] = sub["Forms"].explode().dropna().value_counts().head(3).map(lambda k: n_pct(k, n)).to_dict()

forms_tab = pd.DataFrame(rows).set_index("Drug class")
print("Pharmaceutical form groups by drug class, n (% of approvals in class)")
display(forms_tab)
print("\nMost frequent single forms by drug class, n (% of approvals in class)")
for k, v in top_raw.items():
    print(f"  {k}: {v}")
RESULTS["forms_tab"] = forms_tab

Pharmaceutical form groups by drug class, n (% of approvals in class)


,n approvals,tablet / capsule,solution / injection,powder / lyophilisate,suspension,emulsion / dispersion
Drug class,,,,,,
Small molecules,1764,"1,303 (73.9%)",194 (11.0%),165 (9.4%),24 (1.4%),3 (0.2%)
Biologics,866,10 (1.2%),624 (72.1%),187 (21.6%),0 (0.0%),0 (0.0%)
Peptides and proteins,177,12 (6.8%),103 (58.2%),41 (23.2%),4 (2.3%),2 (1.1%)
Cell and gene therapy,40,0 (0.0%),13 (32.5%),0 (0.0%),3 (7.5%),21 (52.5%)
Biologics + peptides/proteins + CGT,1083,22 (2.0%),740 (68.3%),228 (21.1%),7 (0.6%),23 (2.1%)
Vaccines,207,0 (0.0%),45 (21.7%),27 (13.0%),123 (59.4%),11 (5.3%)



Most frequent single forms by drug class, n (% of approvals in class)
  Small molecules: {'tablet': '993 (56.3%)', 'capsule': '320 (18.1%)', 'powder': '161 (9.1%)'}
  Biologics: {'solution': '418 (48.3%)', 'powder': '185 (21.4%)', 'concentrate': '149 (17.2%)'}
  Peptides and proteins: {'solution': '98 (55.4%)', 'powder': '41 (23.2%)', 'solvent': '13 (7.3%)'}
  Cell and gene therapy: {'dispersion': '21 (52.5%)', 'solution': '6 (15.0%)', 'concentrate': '5 (12.5%)'}
  Biologics + peptides/proteins + CGT: {'solution': '522 (48.2%)', 'powder': '226 (20.9%)', 'concentrate': '155 (14.3%)'}
  Vaccines: {'suspension': '123 (59.4%)', 'powder': '27 (13.0%)', 'solution': '26 (12.6%)'}


## Overlapping drug names across agencies (case-insensitive match)

In [13]:
# Load and clean data
df = pd.read_csv("./../data/datasets/1995/all_decisions/Overall.csv")

df = df[["Non_proprietary_name", "Drug_name", "Dataset"]]

# Exclude Japan / PMDA records
EXCLUDED_DATASETS = {"pmda", "japan"}
df = df[~df["Dataset"].astype(str).str.strip().str.lower().isin(EXCLUDED_DATASETS)]

df["Non_proprietary_name"] = (
    df["Non_proprietary_name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df["Drug_name"] = (
    df["Drug_name"]
    .astype(str)
    .str.strip()
)

df = df.drop_duplicates()

# Wide table: all drug names per INN per agency, then count unique drug names across agencies
grouped = (
    df
    .groupby(["Non_proprietary_name", "Dataset"])["Drug_name"]
    .apply(lambda x: sorted(set(x)))
    .reset_index()
)

pivot = grouped.pivot(
    index="Non_proprietary_name",
    columns="Dataset",
    values="Drug_name"
)

pivot["n_unique_drug_names"] = pivot.apply(
    lambda row: len(set(
        name
        for cell in row.dropna()
        for name in cell
    )),
    axis=1
)

pivot.to_csv(
    "./../output/overlap/drug_names_per_Substance_per_agency.csv"
)

# Number of unique agencies per INN
agency_count = (
    df
    .groupby("Non_proprietary_name")["Dataset"]
    .nunique()
    .rename("n_agencies")
    .reset_index()
)

# Drug name consistency: for each (INN, Drug_name) pair, count in how many agencies this pairing appears
name_agency_map = (
    df
    .groupby(["Non_proprietary_name", "Drug_name"])["Dataset"]
    .nunique()
    .reset_index(name="n_agencies_per_name")
)

# Globally consistent name = the drug name is used by all agencies that report on this INN
name_agency_map = name_agency_map.merge(
    agency_count,
    on="Non_proprietary_name",
    how="left"
)

name_agency_map["is_globally_consistent"] = (
    name_agency_map["n_agencies_per_name"]
    == name_agency_map["n_agencies"]
)

# Summary per INN
summary = (
    name_agency_map
    .groupby("Non_proprietary_name")
    .agg(
        n_agencies=("n_agencies", "first"),
        drug_names=("Drug_name", lambda x: sorted(set(x))),
        has_globally_consistent_name=("is_globally_consistent", "any")
    )
    .reset_index()
)

summary.to_csv(
    "./../output/overlap/drug_name_consistency_summary.csv",
    index=False
)

# Detailed mapping of (INN, Drug_name) pairs with agency counts and consistency flag
name_agency_map.to_csv(
    "./../output/overlap/drug_name_origin_by_agency_count.csv",
    index=False
)

n_total = summary.shape[0]
n_consistent = summary["has_globally_consistent_name"].sum()
percent_consistent = (n_consistent / n_total) * 100

print(f"Total number of substances (INNs): {n_total}")
print(f"Substances with at least one globally consistent drug name: {n_consistent}")
print(f"Proportion with globally consistent name: {percent_consistent:.1f}%")

Total number of substances (INNs): 4556
Substances with at least one globally consistent drug name: 4001
Proportion with globally consistent name: 87.8%


# Results 3.2.6: marketing authorisation holders

In [14]:
# Results 3.2.6: marketing authorisation holders
# Fix: the pie chart in src/analysis/analysis_utils.py used the NUMBER OF OTHER HOLDERS instead of their number of
# approvals as size of the "Other" slice, which inflated the top-20 share (72.8%) and Novartis (8.6%).
# Only spelling variants / renamings of the same company are merged (no mergers or acquisitions).
HOLDER_ALIASES = {
    "merck sharp and dohme": "merck sharp & dohme",
    "merck sharpe and dohme": "merck sharp & dohme",
    "merck sharp dohme": "merck sharp & dohme",
    "merck, sharp and dohme": "merck sharp & dohme",
    "glaxo": "glaxosmithkline",
    "glaxo smith kline": "glaxosmithkline",
    "gsk": "glaxosmithkline",
    "sanofi-aventis": "sanofi",
    "janssen-cilag": "janssen",
    "bristol-myers": "bristol-myers squibb",
    "bristol-meyers squibb": "bristol-myers squibb",
    "bristol myers squibb": "bristol-myers squibb",
    "gilead sciences": "gilead",
    "kyowa hakko kirin": "kyowa kirin",
    "samsung bioepis": "samsung",
    "orphan europe": "orphan",
    "emerge health": "emerge",
    "blueprint medicines": "blueprint",
    "bausch and lomb": "bausch & lomb",
    "zenyaku": "zenyaku kogyo",
}

holder_raw = clean(df_approved["Marketing_authorisation_holder_extracted"])
df_approved["Holder"] = holder_raw.replace(HOLDER_ALIASES)

n_all = len(df_approved)
reported = df_approved["Holder"].dropna()
vc = reported.value_counts()
top20 = vc.head(20)

RESULTS["holder_not_reported"] = int(n_all - len(reported))
RESULTS["holder_unique"] = int(len(vc))
RESULTS["holder_top20_share"] = pct(top20.sum(), len(reported))
RESULTS["holder_top20_share_all"] = pct(top20.sum(), n_all)
RESULTS["holder_rest"] = (int(len(vc) - 20), pct(vc.iloc[20:].sum(), len(reported)))
RESULTS["holder_largest"] = (vc.index[0], int(vc.iloc[0]), pct(vc.iloc[0], len(reported)))

top_tab = pd.DataFrame({"approvals": top20, "% of approvals with reported holder": (100 * top20 / len(reported)).round(1)})
display(top_tab)
print(f"Approvals with reported holder: {len(reported):,} of {n_all:,} (not reported: {RESULTS['holder_not_reported']})")
print(f"Unique holders: {RESULTS['holder_unique']}")
print(f"Top 20 share: {RESULTS['holder_top20_share']}% of approvals with reported holder "
      f"({RESULTS['holder_top20_share_all']}% of all approvals)")
print(f"Remaining {RESULTS['holder_rest'][0]} holders: {RESULTS['holder_rest'][1]}%")
print(f"Largest holder: {RESULTS['holder_largest']}")

# Largest holder per agency
print("\nLargest holder per agency")
for a in AGENCIES:
    s = df_approved.loc[df_approved["Agency"] == a, "Holder"].dropna().value_counts()
    print(f"  {a}: {s.index[0]} {n_pct(s.iloc[0], s.sum())}, unique holders {len(s)}")

# Reproduction of the old (incorrect) calculation, without alias normalisation
old_vc = holder_raw.dropna().value_counts()
old_total = old_vc.head(20).sum() + (len(old_vc) - 20)
print(f"\n(Old pie-chart logic) top 20: {pct(old_vc.head(20).sum(), old_total)}%, "
      f"largest: {pct(old_vc.iloc[0], old_total)}%, 'other holders': {len(old_vc) - 20}")

# Correct input for Supplementary Figure 4 (top 20 + Other, sizes = approvals)
pie_data = pd.concat([top20, pd.Series({f"Other ({len(vc) - 20} holders)": vc.iloc[20:].sum()})])

,approvals,% of approvals with reported holder
Holder,,
novartis,164,5.3
pfizer,135,4.4
glaxosmithkline,98,3.2
astrazeneca,97,3.2
roche,93,3.0
janssen,87,2.8
merck sharp & dohme,84,2.7
sanofi,81,2.6
bristol-myers squibb,68,2.2


Approvals with reported holder: 3,076 of 3,120 (not reported: 44)
Unique holders: 518
Top 20 share: 48.0% of approvals with reported holder (47.3% of all approvals)
Remaining 498 holders: 52.0%
Largest holder: ('novartis', 164, 5.3)

Largest holder per agency
  EMA: novartis 68 (4.7%), unique holders 392
  TGA: novartis 67 (6.8%), unique holders 151
  PMDA: chugai 23 (5.6%), unique holders 84
  Swissmedic: roche 14 (6.0%), unique holders 90

(Old pie-chart logic) top 20: 73.0%, largest: 8.6%, 'other holders': 516


# Results 3.2.7 – Regulatory review durations

In [15]:
# Results 3.2.7: regulatory review durations (approved applications, positive durations only)
PROCEDURE_MAP = {"yes": "Abridged", "no": "Non-abridged"}

dur = df_approved.copy()
dur["Review_days"] = (to_date(dur["Decision_date"]) - to_date(dur["Application_date"])).dt.days
dur["Procedure"] = clean(dur["Nonclinical_abridged"]).map(PROCEDURE_MAP)
dur["Valid"] = dur["Review_days"].gt(0) & dur["Procedure"].notna()

excl = dur.groupby("Agency").agg(
    approvals=("Valid", "size"),
    missing_date=("Review_days", lambda x: int(x.isna().sum())),
    non_positive=("Review_days", lambda x: int((x <= 0).sum())),
    missing_procedure=("Procedure", lambda x: int(x.isna().sum())),
    analysed=("Valid", "sum"),
).reindex(AGENCIES)
print("Review duration: available data")
display(excl)
short = dur[dur["Valid"] & dur["Review_days"].lt(30)]
print(f"Plausibility check - review durations < 30 days: {len(short)} {short['Agency'].value_counts().to_dict()}")
RESULTS["dur_available"] = excl


def summarise(x: pd.Series) -> pd.Series:
    q1, q3 = x.quantile(0.25), x.quantile(0.75)
    return pd.Series({"n": len(x), "median": x.median(), "Q1": q1, "Q3": q3, "IQR": q3 - q1,
                      "min": x.min(), "max": x.max()})


valid = dur[dur["Valid"]]
by_agency_proc = valid.groupby(["Agency", "Procedure"])["Review_days"].apply(summarise).unstack()
by_agency_proc = by_agency_proc.reindex(pd.MultiIndex.from_product([AGENCIES, ["Non-abridged", "Abridged"]]))
print("\nReview duration (days) by agency and procedure")
display(by_agency_proc)

by_proc = valid.groupby("Procedure")["Review_days"].apply(summarise).unstack()
print("\nOverall by procedure")
display(by_proc)

RESULTS["dur"] = by_agency_proc
for proc in ["Non-abridged", "Abridged"]:
    t = by_agency_proc.xs(proc, level=1)
    RESULTS[f"dur_median_rank_{proc}"] = t["median"].sort_values(ascending=False).astype(int).to_dict()
    RESULTS[f"dur_iqr_rank_{proc}"] = t["IQR"].sort_values().round(0).astype(int).to_dict()
    print(f"\n{proc}: median ranking (longest first) {RESULTS[f'dur_median_rank_{proc}']}")
    print(f"{proc}: IQR ranking (least variable first) {RESULTS[f'dur_iqr_rank_{proc}']}")

# Figure 4B: pooled across agencies, by disease class
dur_cls = valid[["Review_days", "Disease_classes"]].explode("Disease_classes").dropna()
by_class = dur_cls.groupby("Disease_classes")["Review_days"].apply(summarise).unstack().sort_values("median")
print("\nReview duration (days) by disease class, pooled across agencies")
display(by_class[["n", "median", "Q1", "Q3", "IQR"]])
RESULTS["dur_class_median_range"] = (int(by_class.loc[by_class["n"] >= 20, "median"].min()),
                                     int(by_class.loc[by_class["n"] >= 20, "median"].max()))

Review duration: available data


,approvals,missing_date,non_positive,missing_procedure,analysed
Agency,,,,,
EMA,1491,95,0,1,1396
TGA,988,479,10,0,499
PMDA,408,4,1,0,403
Swissmedic,233,0,0,0,233


Plausibility check - review durations < 30 days: 23 {'EMA': 11, 'PMDA': 9, 'TGA': 3}

Review duration (days) by agency and procedure


n  median      Q1      Q3     IQR    min     max
EMA        Non-abridged  907.0   378.0  329.00  454.00  125.00    2.0  1627.0
           Abridged      489.0   331.0  254.00  398.00  144.00    9.0  1004.0
TGA        Non-abridged  296.0   350.0  299.00  394.75   95.75   32.0  3284.0
           Abridged      203.0   352.0  293.50  387.50   94.00    4.0  2388.0
PMDA       Non-abridged  260.0   269.0  218.00  320.00  102.00   20.0  1188.0
           Abridged      143.0   244.0  198.00  289.00   91.00    3.0   750.0
Swissmedic Non-abridged   77.0   374.0  294.00  493.00  199.00  109.0   714.0
           Abridged      156.0   405.0  265.75  494.25  228.50   57.0   865.0


Overall by procedure


,n,median,Q1,Q3,IQR,min,max
Procedure,,,,,,,
Abridged,991.0,324.0,241.5,399.50,158.00,3.0,2388.0
Non-abridged,1540.0,357.0,287.0,432.25,145.25,2.0,3284.0



Non-abridged: median ranking (longest first) {'EMA': 378, 'Swissmedic': 374, 'TGA': 350, 'PMDA': 269}
Non-abridged: IQR ranking (least variable first) {'TGA': 96, 'PMDA': 102, 'EMA': 125, 'Swissmedic': 199}

Abridged: median ranking (longest first) {'Swissmedic': 405, 'TGA': 352, 'EMA': 331, 'PMDA': 244}
Abridged: IQR ranking (least variable first) {'PMDA': 91, 'TGA': 94, 'EMA': 144, 'Swissmedic': 228}

Review duration (days) by disease class, pooled across agencies


,n,median,Q1,Q3,IQR
Disease_classes,,,,,
infectious and parasitic diseases,424.0,317.5,175.00,393.00,218.00
diseases of the respiratory system,319.0,333.0,238.00,387.00,149.00
diseases of the circulatory system,189.0,335.0,258.00,414.00,156.00
diseases of the musculoskeletal system and connective tissue,196.0,349.5,299.50,420.75,121.25
neoplasms,749.0,351.0,270.00,419.00,149.00
diseases of the skin,193.0,353.0,288.00,412.00,124.00
congenital malformations and chromosomal abnormalities,29.0,353.0,266.00,400.00,134.00
diseases of the nervous system,260.0,355.5,288.00,426.00,138.00
diseases of the blood and blood-forming organs,432.0,356.0,280.75,427.25,146.50
